> **Versión GitHub reproducible.** Esta copia no modifica el notebook original de Google Drive. Las rutas personales de Drive se han sustituido por rutas del repositorio. Los datos brutos de gran tamaño no incluidos en GitHub deben descargarse según `docs/DATA_SOURCES.md`.


In [ ]:
# ============================================================
# CONFIGURACIÓN DE RUTAS - VERSIÓN GITHUB / GOOGLE COLAB
# ============================================================
# Clone primero el repositorio con:
# !git clone https://github.com/ritaridh/tfm-crop-yield-prediction.git /content/TFM_Rendimiento_cultivos
# %cd /content/TFM_Rendimiento_cultivos

from pathlib import Path
import os

ROOT = Path("/content/TFM_Rendimiento_cultivos")
if not ROOT.exists():
    raise FileNotFoundError(
        "No se encuentra el repositorio. Ejecute primero el bloque de clonación indicado en el README."
    )

RUTA_TFM = str(ROOT)
RUTA_BASE = str(ROOT)
BASE = str(ROOT)
BASE_DIR = ROOT
RUTA_ORIGINALES = str(ROOT / "01_Datos_originales")
RUTA_EUROCROPS = str(ROOT / "01_Datos_originales" / "EuroCrops")
RUTA_MAPA = str(ROOT / "01_Datos_originales" / "MAPA")
RUTA_SENTINEL = str(ROOT / "01_Datos_originales" / "Sentinel2")
RUTA_PROCESADOS = str(ROOT / "02_Datos_procesados")
RUTA_RESULTADOS = str(ROOT / "05_Resultados")
PROC = RUTA_PROCESADOS
RESULTADOS = RUTA_RESULTADOS

Path(RUTA_PROCESADOS).mkdir(parents=True, exist_ok=True)
Path(RUTA_RESULTADOS).mkdir(parents=True, exist_ok=True)
print("Repositorio:", ROOT)


In [ ]:
# ============================================================
# PASO 2 - DEFINIR RUTAS DEL PROYECTO
# ============================================================

import os

RUTA_BASE = (
    "/content/TFM_Rendimiento_cultivos/"
    "TFM_Rendimiento_cultivos"
)

RUTA_ORIGINALES = os.path.join(
    RUTA_BASE,
    "01_Datos_originales"
)

RUTA_PROCESADOS = os.path.join(
    RUTA_BASE,
    "02_Datos_procesados"
)

RUTA_EUROCROPS = os.path.join(
    RUTA_ORIGINALES,
    "EuroCrops"
)

RUTA_SENTINEL = os.path.join(
    RUTA_ORIGINALES,
    "Sentinel2"
)

RUTA_RESULTADOS = os.path.join(
    RUTA_BASE,
    "05_Resultados"
)

# Crear carpeta Sentinel-2 si todavía no existe
os.makedirs(
    RUTA_SENTINEL,
    exist_ok=True
)

print("RUTA BASE:")
print(RUTA_BASE)

print("\nEuroCrops:")
print(RUTA_EUROCROPS)

print("\nSentinel-2:")
print(RUTA_SENTINEL)

print("\nProcesados:")
print(RUTA_PROCESADOS)

# EuroCrops

In [ ]:
# ============================================================
# PASO 3 - COMPROBAR CONTENIDO DE EUROCROPS
# ============================================================

print(
    "Existe carpeta EuroCrops:",
    os.path.exists(RUTA_EUROCROPS)
)

print("\nContenido principal:\n")

for elemento in sorted(
    os.listdir(RUTA_EUROCROPS)
):
    print(elemento)

In [ ]:
# ============================================================
# PASO 4 - LOCALIZAR SHAPEFILES EUROCROPS 2023
# ============================================================

import glob

shapefiles_2023 = glob.glob(
    os.path.join(
        RUTA_EUROCROPS,
        "**",
        "*2023*.shp"
    ),
    recursive=True
)

print(
    "Número de shapefiles 2023 encontrados:",
    len(shapefiles_2023)
)

print("\nPrimeros 10:")

for ruta in sorted(shapefiles_2023)[:10]:
    print(ruta)

In [ ]:
# ============================================================
# PASO 5 - INSPECCIONAR UN SHAPEFILE EUROCROPS
# ============================================================

import geopandas as gpd

ruta_prueba = sorted(
    shapefiles_2023
)[0]

print("Archivo de prueba:")
print(ruta_prueba)

euro_prueba = gpd.read_file(
    ruta_prueba
)

print("\nDimensiones:")
print(euro_prueba.shape)

print("\nCRS:")
print(euro_prueba.crs)

print("\nColumnas:")
print(euro_prueba.columns.tolist())

print("\nPrimeras filas:")

display(
    euro_prueba.head()
)

In [ ]:
# ============================================================
# PASO 6 - INSPECCIONAR CLASIFICACIÓN HCAT
# SIN CARGAR GEOMETRÍAS
# ============================================================

import geopandas as gpd
import pandas as pd

hcat_prueba = gpd.read_file(
    ruta_prueba,
    columns=[
        "EC_hcat_n",
        "EC_hcat_c"
    ],
    ignore_geometry=True
)

print("Dimensiones:")
print(hcat_prueba.shape)

print("\nNúmero de categorías HCAT:")
print(hcat_prueba["EC_hcat_n"].nunique())

print("\nPrimeras categorías:")
display(
    hcat_prueba[
        ["EC_hcat_n", "EC_hcat_c"]
    ]
    .drop_duplicates()
    .sort_values("EC_hcat_n")
    .head(50)
)

In [ ]:
# ============================================================
# PASO 7 - BUSCAR TRIGO BLANDO Y CEBADA EN HCAT
# ============================================================

categorias_hcat = (
    hcat_prueba[
        ["EC_hcat_n", "EC_hcat_c"]
    ]
    .drop_duplicates()
)

mask_cereales = (
    categorias_hcat["EC_hcat_n"]
    .astype(str)
    .str.contains(
        "wheat|barley|trigo|cebada",
        case=False,
        na=False
    )
)

display(
    categorias_hcat[
        mask_cereales
    ].sort_values("EC_hcat_n")
)

In [ ]:
# ============================================================
# PASO 8 - INVENTARIO EUROCROPS 2023
# TRIGO BLANDO Y CEBADA EN LAS 50 PROVINCIAS
# ============================================================

import geopandas as gpd
import pandas as pd
import os

cultivos_objetivo = [
    "barley",
    "common_soft_wheat"
]

resultados = []

for i, shp in enumerate(shapefiles_2023, start=1):

    print(f"[{i:02d}/{len(shapefiles_2023)}] {os.path.basename(shp)}")

    # Leer solo columnas necesarias
    df = gpd.read_file(
        shp,
        columns=[
            "provincia",
            "exp_prov",
            "EC_hcat_n",
            "EC_hcat_c",
            "par_supcul"
        ],
        ignore_geometry=True
    )

    # Filtrar cultivos objetivo
    df_cultivos = df[
        df["EC_hcat_n"].isin(cultivos_objetivo)
    ].copy()

    # Información básica del archivo
    provincia_codigo = (
        df["provincia"].iloc[0]
        if len(df) > 0
        else None
    )

    exp_prov = (
        df["exp_prov"].iloc[0]
        if len(df) > 0
        else None
    )

    for cultivo in cultivos_objetivo:

        sub = df_cultivos[
            df_cultivos["EC_hcat_n"] == cultivo
        ]

        resultados.append({
            "archivo": os.path.basename(shp),
            "provincia_codigo": provincia_codigo,
            "exp_prov": exp_prov,
            "cultivo": cultivo,
            "n_recintos": len(sub),
            "par_supcul_suma": pd.to_numeric(
                sub["par_supcul"],
                errors="coerce"
            ).sum()
        })

inventario_eurocrops = pd.DataFrame(resultados)

print("\nProceso terminado.")
print("Dimensiones:", inventario_eurocrops.shape)

display(inventario_eurocrops.head(20))

In [ ]:
# ============================================================
# PASO 9 - RESUMEN NACIONAL EUROCROPS 2023
# ============================================================

resumen_nacional = (
    inventario_eurocrops
    .groupby("cultivo")
    .agg(
        n_recintos=("n_recintos", "sum"),
        par_supcul_suma=("par_supcul_suma", "sum"),
        provincias_con_cultivo=(
            "n_recintos",
            lambda x: (x > 0).sum()
        )
    )
    .reset_index()
)

display(resumen_nacional)

print("\nNúmero esperado de registros del inventario:")
print(50 * 2)

print("\nNúmero obtenido:")
print(len(inventario_eurocrops))

print("\nProvincias sin cebada:")
display(
    inventario_eurocrops[
        (inventario_eurocrops["cultivo"] == "barley") &
        (inventario_eurocrops["n_recintos"] == 0)
    ]
)

print("\nProvincias sin trigo blando:")
display(
    inventario_eurocrops[
        (inventario_eurocrops["cultivo"] == "common_soft_wheat") &
        (inventario_eurocrops["n_recintos"] == 0)
    ]
)

In [ ]:
# Comprobaciones

In [ ]:
# ============================================================
# PASO 10 - COMPROBAR UNIDAD DE par_supcul
# Comparación con superficie geométrica
# ============================================================

import geopandas as gpd
import pandas as pd
import numpy as np

# Leer archivo de prueba completo
gdf_test = gpd.read_file(ruta_prueba)

# Quedarnos solo con los dos cultivos
gdf_test = gdf_test[
    gdf_test["EC_hcat_n"].isin([
        "barley",
        "common_soft_wheat"
    ])
].copy()

print("Recintos seleccionados:", len(gdf_test))
print("CRS original:", gdf_test.crs)

# Convertir par_supcul a numérico
gdf_test["par_supcul"] = pd.to_numeric(
    gdf_test["par_supcul"],
    errors="coerce"
)

# ------------------------------------------------------------
# Para calcular áreas NO usamos EPSG:4258.
# Reproyectamos a un CRS métrico.
# Para Alicante usamos ETRS89 / UTM zone 30N.
# ------------------------------------------------------------

gdf_test_utm = gdf_test.to_crs("EPSG:25830")

# Superficie geométrica
gdf_test_utm["area_geom_m2"] = gdf_test_utm.geometry.area
gdf_test_utm["area_geom_ha"] = (
    gdf_test_utm["area_geom_m2"] / 10000
)

# Inspección
columnas_control = [
    "EC_hcat_n",
    "par_supcul",
    "area_geom_m2",
    "area_geom_ha"
]

display(
    gdf_test_utm[columnas_control].head(20)
)

print("\nResumen:")
display(
    gdf_test_utm[
        ["par_supcul", "area_geom_m2", "area_geom_ha"]
    ].describe()
)

In [ ]:
# ============================================================
# PASO 11 - RELACIÓN par_supcul / SUPERFICIE GEOMÉTRICA
# ============================================================

control_area = gdf_test_utm[
    [
        "EC_hcat_n",
        "par_supcul",
        "area_geom_m2",
        "area_geom_ha"
    ]
].copy()

# Evitar divisiones por cero
control_area = control_area[
    (control_area["par_supcul"] > 0) &
    (control_area["area_geom_m2"] > 0)
].copy()

control_area["ratio_par_supcul_m2"] = (
    control_area["par_supcul"] /
    control_area["area_geom_m2"]
)

control_area["ratio_par_supcul_ha"] = (
    control_area["par_supcul"] /
    control_area["area_geom_ha"]
)

print("Número de recintos comparables:")
print(len(control_area))

print("\nRatio par_supcul / área geométrica en m²:")
display(
    control_area["ratio_par_supcul_m2"].describe()
)

print("\nRatio par_supcul / área geométrica en ha:")
display(
    control_area["ratio_par_supcul_ha"].describe()
)

print("\nPor cultivo:")
display(
    control_area.groupby("EC_hcat_n")[
        [
            "par_supcul",
            "area_geom_m2",
            "area_geom_ha",
            "ratio_par_supcul_m2"
        ]
    ].median()
)

In [ ]:
# Construir Eurocrops provincial 2023

In [ ]:
# ============================================================
# PASO 12 - RESUMEN PROVINCIAL EUROCROPS 2023
# ============================================================

eurocrops_prov_2023 = inventario_eurocrops.copy()

# Superficie declarada: m² -> ha
eurocrops_prov_2023["superficie_eurocrops_ha"] = (
    eurocrops_prov_2023["par_supcul_suma"] / 10000
)

# Mantener únicamente columnas útiles
eurocrops_prov_2023 = eurocrops_prov_2023[
    [
        "archivo",
        "provincia_codigo",
        "exp_prov",
        "cultivo",
        "n_recintos",
        "superficie_eurocrops_ha"
    ]
].copy()

print("Dimensiones:", eurocrops_prov_2023.shape)

print("\nRegistros por cultivo:")
print(
    eurocrops_prov_2023["cultivo"].value_counts()
)

print("\nResumen nacional:")
display(
    eurocrops_prov_2023
    .groupby("cultivo")
    .agg(
        n_recintos=("n_recintos", "sum"),
        superficie_eurocrops_ha=("superficie_eurocrops_ha", "sum"),
        provincias_con_cultivo=("n_recintos", lambda x: (x > 0).sum())
    )
)

In [ ]:
# ============================================================
# PASO 13 - CARGAR PROVINCIAS DESDE DRIVE
# ============================================================

import geopandas as gpd
import os

ruta_provincias = os.path.join(
    RUTA_PROCESADOS,
    "provincias_CNIG_50.gpkg"
)

provincias_final = gpd.read_file(
    ruta_provincias,
    layer="provincias"
)

print("Dimensiones:", provincias_final.shape)
print("CRS:", provincias_final.crs)

print("\nColumnas:")
print(provincias_final.columns.tolist())

display(
    provincias_final.drop(
        columns="geometry",
        errors="ignore"
    ).head(15)
)

In [ ]:
# ============================================================
# PASO 14 - MAPA CÓDIGO PROVINCIAL -> NOMBRE ESTÁNDAR
# ============================================================

codigo_provincia_nombre = {
    1: "Álava",
    2: "Albacete",
    3: "Alicante",
    4: "Almería",
    5: "Ávila",
    6: "Badajoz",
    7: "Illes Balears",
    8: "Barcelona",
    9: "Burgos",
    10: "Cáceres",
    11: "Cádiz",
    12: "Castellón",
    13: "Ciudad Real",
    14: "Córdoba",
    15: "A Coruña",
    16: "Cuenca",
    17: "Girona",
    18: "Granada",
    19: "Guadalajara",
    20: "Gipuzkoa",
    21: "Huelva",
    22: "Huesca",
    23: "Jaén",
    24: "León",
    25: "Lleida",
    26: "La Rioja",
    27: "Lugo",
    28: "Madrid",
    29: "Málaga",
    30: "Murcia",
    31: "Navarra",
    32: "Ourense",
    33: "Asturias",
    34: "Palencia",
    35: "Las Palmas",
    36: "Pontevedra",
    37: "Salamanca",
    38: "Santa Cruz de Tenerife",
    39: "Cantabria",
    40: "Segovia",
    41: "Sevilla",
    42: "Soria",
    43: "Tarragona",
    44: "Teruel",
    45: "Toledo",
    46: "Valencia",
    47: "Valladolid",
    48: "Bizkaia",
    49: "Zamora",
    50: "Zaragoza"
}

print("Número de códigos:", len(codigo_provincia_nombre))

In [ ]:
# ============================================================
# PASO 15
# IDENTIFICAR CADA SHAPEFILE EUROCROPS MEDIANTE CNIG
# ============================================================

import geopandas as gpd
import pandas as pd
import os

correspondencias = []

for i, shp in enumerate(shapefiles_2023, start=1):

    print(
        f"[{i:02d}/{len(shapefiles_2023)}] "
        f"{os.path.basename(shp)}"
    )

    # Leer únicamente geometría
    gdf_tmp = gpd.read_file(
        shp,
        columns=[],
    )

    if len(gdf_tmp) == 0:
        correspondencias.append({
            "archivo": os.path.basename(shp),
            "provincia_estandar": None
        })
        continue

    # Pasar al mismo CRS que CNIG
    gdf_tmp = gdf_tmp.to_crs(
        provincias_final.crs
    )

    # Crear un punto representativo de un recinto
    punto = gdf_tmp.geometry.iloc[0].representative_point()

    # Buscar provincia CNIG que contiene el punto
    coincidencia = provincias_final[
        provincias_final.geometry.covers(punto)
    ]

    if len(coincidencia) == 1:

        nombre = coincidencia.iloc[0][
            "provincia_estandar"
        ]

    else:
        nombre = None

    correspondencias.append({
        "archivo": os.path.basename(shp),
        "provincia_estandar": nombre
    })

correspondencia_euro_cnig = pd.DataFrame(
    correspondencias
)

print("\n==============================")
print("CONTROL")
print("==============================")

print(
    "Shapefiles:",
    len(correspondencia_euro_cnig)
)

print(
    "Provincias identificadas:",
    correspondencia_euro_cnig[
        "provincia_estandar"
    ].nunique()
)

print(
    "Sin identificar:",
    correspondencia_euro_cnig[
        "provincia_estandar"
    ].isna().sum()
)

display(
    correspondencia_euro_cnig.sort_values(
        "provincia_estandar"
    )
)

In [ ]:
# ============================================================
# PASO 16 - ASOCIAR EUROCROPS CON PROVINCIA ESTÁNDAR
# ============================================================

eurocrops_prov_2023 = eurocrops_prov_2023.drop(
    columns=["provincia_estandar"],
    errors="ignore"
)

eurocrops_prov_2023 = eurocrops_prov_2023.merge(
    correspondencia_euro_cnig,
    on="archivo",
    how="left",
    validate="many_to_one"
)

print(
    "Registros:",
    len(eurocrops_prov_2023)
)

print(
    "Provincias únicas:",
    eurocrops_prov_2023["provincia_estandar"].nunique()
)

print(
    "Sin provincia:",
    eurocrops_prov_2023["provincia_estandar"].isna().sum()
)

display(
    eurocrops_prov_2023[
        [
            "provincia_estandar",
            "cultivo",
            "n_recintos",
            "superficie_eurocrops_ha"
        ]
    ]
    .sort_values(
        ["provincia_estandar", "cultivo"]
    )
    .head(20)
)

In [ ]:
# ============================================================
# PASO 17 - CONTROL FINAL EUROCROPS
# ============================================================

print("Duplicados provincia × cultivo:")

print(
    eurocrops_prov_2023.duplicated(
        subset=[
            "provincia_estandar",
            "cultivo"
        ]
    ).sum()
)

print("\nBizkaia:")

display(
    eurocrops_prov_2023[
        eurocrops_prov_2023[
            "provincia_estandar"
        ] == "Bizkaia"
    ]
)

print("\nProvincias con n_recintos = 0:")

display(
    eurocrops_prov_2023[
        eurocrops_prov_2023[
            "n_recintos"
        ] == 0
    ][
        [
            "provincia_estandar",
            "cultivo",
            "n_recintos",
            "superficie_eurocrops_ha"
        ]
    ]
)

In [ ]:
# ============================================================
# PASO 18 - VARIABLES ESTRUCTURALES EUROCROPS 2023
# ============================================================

eurocrops_prov_2023[
    "superficie_media_recinto_ha"
] = np.where(
    eurocrops_prov_2023["n_recintos"] > 0,
    eurocrops_prov_2023["superficie_eurocrops_ha"] /
    eurocrops_prov_2023["n_recintos"],
    0
)

display(
    eurocrops_prov_2023[
        [
            "provincia_estandar",
            "cultivo",
            "n_recintos",
            "superficie_eurocrops_ha",
            "superficie_media_recinto_ha"
        ]
    ].head(20)
)

In [ ]:
# ============================================================
# PASO 19 - GUARDAR EUROCROPS 2023 DEFINITIVO
# ============================================================

ruta_eurocrops_final = os.path.join(
    RUTA_PROCESADOS,
    "EuroCrops_2023_trigo_cebada_provincial.csv"
)

eurocrops_final = (
    eurocrops_prov_2023[
        [
            "provincia_estandar",
            "cultivo",
            "n_recintos",
            "superficie_eurocrops_ha",
            "superficie_media_recinto_ha"
        ]
    ]
    .copy()
    .sort_values(
        ["provincia_estandar", "cultivo"]
    )
    .reset_index(drop=True)
)

eurocrops_final.to_csv(
    ruta_eurocrops_final,
    index=False,
    encoding="utf-8-sig"
)

print("Guardado:")
print(ruta_eurocrops_final)

print("\nDimensiones:")
print(eurocrops_final.shape)

print("\nNulos:")
print(eurocrops_final.isna().sum().sum())

In [ ]:
# ============================================================
# PASO 20 - CONTROL FINAL ANTES DEL BLOQUE SENTINEL-2
# ============================================================

import os
import pandas as pd

print("ARCHIVOS PROCESADOS DISPONIBLES:\n")

for archivo in sorted(os.listdir(RUTA_PROCESADOS)):
    print(" -", archivo)

print("\n" + "=" * 60)
print("CONTROL DE LOS TRES BLOQUES PREPARADOS")
print("=" * 60)

archivos_clave = [
    "target_trigo_cebada_2017_2023.csv",
    "SoilGrids_provincial_0_30cm.csv",
    "EuroCrops_2023_trigo_cebada_provincial.csv"
]

for archivo in archivos_clave:
    ruta = os.path.join(RUTA_PROCESADOS, archivo)

    print(f"\n{archivo}")
    print("Existe:", os.path.exists(ruta))

    if os.path.exists(ruta):
        df = pd.read_csv(ruta)
        print("Dimensiones:", df.shape)
        print("Columnas:", df.columns.tolist())

# SENTINEL-2

In [ ]:
# ============================================================
# PASO 21 - INSTALAR EARTH ENGINE (Preparar Google Earth Engine en Colab)
# ============================================================

!pip -q install earthengine-api geemap

In [ ]:
# ============================================================
# PASO 22 - AUTENTICAR EARTH ENGINE
# ============================================================

import ee

ee.Authenticate()

In [ ]:
# ============================================================
# PASO 23A - COMPROBAR PROYECTOS DE GOOGLE CLOUD
# ============================================================

!gcloud projects list

In [ ]:
# ============================================================
# PASO 23C - INICIALIZAR EARTH ENGINE CON EL PROYECTO
# ============================================================

import ee

PROJECT_ID = "tfm-rendimiento-cultivos"

ee.Initialize(
    project=PROJECT_ID
)

print("Google Earth Engine inicializado correctamente.")
print("Proyecto:", PROJECT_ID)

In [ ]:
# ============================================================
# PASO 24 - COMPROBAR SENTINEL-2 SR HARMONIZED EN 2017
# ============================================================

coleccion_s2_2017 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterDate("2017-01-01", "2018-01-01")
)

n_imagenes_2017 = coleccion_s2_2017.size().getInfo()

print(
    "Número de imágenes Sentinel-2 SR en 2017:",
    n_imagenes_2017
)

In [ ]:
# ============================================================
# PASO 25 - CARGAR PROVINCIAS CNIG
# ============================================================

import geopandas as gpd
import os

ruta_provincias = os.path.join(
    RUTA_PROCESADOS,
    "provincias_CNIG_50.gpkg"
)

provincias = gpd.read_file(
    ruta_provincias,
    layer="provincias"
)

provincias = provincias.to_crs("EPSG:4326")

print("Provincias:", len(provincias))
print("CRS:", provincias.crs)
print("Geometrías inválidas:", (~provincias.geometry.is_valid).sum())

In [ ]:
# ============================================================
# PASO 26 - GEOMETRÍA DE ESPAÑA PARA EARTH ENGINE
# ============================================================

# Unir las 50 provincias
espana_geom = provincias.geometry.union_all()

# Simplificación ligera únicamente para filtro espacial
espana_geom_simple = espana_geom.simplify(
    tolerance=0.01,
    preserve_topology=True
)

# Convertir a GeoJSON
geojson_espana = espana_geom_simple.__geo_interface__

# Convertir a geometría Earth Engine
ee_espana = ee.Geometry(geojson_espana)

print("Geometría de España preparada para Earth Engine.")

In [ ]:
# ============================================================
# PASO 27 - SENTINEL-2 2017 SOBRE ESPAÑA
# ============================================================

s2_espana_2017 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(ee_espana)
    .filterDate("2017-01-01", "2018-01-01")
)

n_espana_2017 = s2_espana_2017.size().getInfo()

print(
    "Imágenes Sentinel-2 SR 2017 que intersectan España:",
    n_espana_2017
)

In [ ]:
# ============================================================
# PASO 28 - DISPONIBILIDAD SENTINEL-2 POR AÑO
# ============================================================

disponibilidad_s2 = []

for ano in range(2017, 2024):

    coleccion = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(ee_espana)
        .filterDate(
            f"{ano}-01-01",
            f"{ano + 1}-01-01"
        )
    )

    n = coleccion.size().getInfo()

    disponibilidad_s2.append({
        "ano": ano,
        "n_imagenes": n
    })

    print(ano, "->", n)

#Prueba con Valladolid

In [ ]:
# ============================================================
# PASO 29 - CREAR GEOMETRÍA PILOTO: VALLADOLID
# ============================================================

provincia_piloto = "Valladolid"

gdf_piloto = provincias[
    provincias["provincia_estandar"] == provincia_piloto
].copy()

print("Geometrías encontradas:", len(gdf_piloto))

# Unir geometrías por seguridad
geom_piloto = gdf_piloto.geometry.union_all()

# Simplificación ligera
geom_piloto_simple = geom_piloto.simplify(
    tolerance=0.001,
    preserve_topology=True
)

# Convertir a Earth Engine
ee_provincia_piloto = ee.Geometry(
    geom_piloto_simple.__geo_interface__
)

print("Provincia piloto:", provincia_piloto)
print("Geometría preparada correctamente.")

In [ ]:
# ============================================================
# PASO 30 - SENTINEL-2 PARA VALLADOLID - CAMPAÑA 2023
# ============================================================

fecha_inicio = "2022-09-01"
fecha_fin = "2023-09-01"

s2_piloto = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(ee_provincia_piloto)
    .filterDate(fecha_inicio, fecha_fin)
)

n_imagenes = s2_piloto.size().getInfo()

print("Provincia:", provincia_piloto)
print("Campaña:", fecha_inicio, "a", fecha_fin)
print("Imágenes Sentinel-2 encontradas:", n_imagenes)

In [ ]:
# ============================================================
# PASO 31 - DISTRIBUCIÓN DE NUBOSIDAD
# ============================================================

cloud_values = (
    s2_piloto
    .aggregate_array("CLOUDY_PIXEL_PERCENTAGE")
    .getInfo()
)

import pandas as pd

cloud_series = pd.Series(
    cloud_values,
    name="CLOUDY_PIXEL_PERCENTAGE"
)

print("Número de imágenes:", len(cloud_series))

print("\nResumen de nubosidad:")
display(
    cloud_series.describe(
        percentiles=[0.25, 0.50, 0.75, 0.90]
    )
)

In [ ]:
# ============================================================
# PASO 32 - EVALUAR UMBRALES DE NUBOSIDAD
# ============================================================

umbrales = [10, 20, 30, 40, 50, 60, 80]

resultados_nubes = []

for umbral in umbrales:

    n = (
        s2_piloto
        .filter(
            ee.Filter.lte(
                "CLOUDY_PIXEL_PERCENTAGE",
                umbral
            )
        )
        .size()
        .getInfo()
    )

    resultados_nubes.append({
        "umbral_nubes_pct": umbral,
        "n_imagenes": n
    })

tabla_nubes = pd.DataFrame(resultados_nubes)

display(tabla_nubes)

In [ ]:
# ============================================================
# PASO 33 - LOCALIZAR EUROCROPS 2023 DE VALLADOLID
# ============================================================

import glob
import os

patron_va = os.path.join(
    RUTA_EUROCROPS,
    "ES_2023",
    "**",
    "es_va_2023.shp"
)

archivos_va = glob.glob(
    patron_va,
    recursive=True
)

print("Archivos encontrados:", len(archivos_va))

for archivo in archivos_va:
    print(archivo)

In [ ]:
# ============================================================
# PASO 34 - CARGAR TRIGO Y CEBADA DE VALLADOLID
# ============================================================

ruta_va = archivos_va[0]

columnas_necesarias = [
    "EC_hcat_n",
    "par_supcul",
    "geometry"
]

va = gpd.read_file(
    ruta_va,
    columns=columnas_necesarias
)

cultivos_objetivo = [
    "barley",
    "common_soft_wheat"
]

va_cereales = va[
    va["EC_hcat_n"].isin(cultivos_objetivo)
].copy()

print("CRS:", va_cereales.crs)
print("Recintos seleccionados:", len(va_cereales))

print("\nRecintos por cultivo:")
print(
    va_cereales["EC_hcat_n"]
    .value_counts()
)

print("\nSuperficie declarada (ha):")

superficie = (
    va_cereales
    .groupby("EC_hcat_n")["par_supcul"]
    .sum()
    / 10000
)

print(superficie)

In [ ]:
# ============================================================
# PASO 35 - CONTROL GEOMÉTRICO
# ============================================================

print("Número de geometrías:", len(va_cereales))
print("Geometrías nulas:", va_cereales.geometry.isna().sum())
print("Geometrías vacías:", va_cereales.geometry.is_empty.sum())
print("Geometrías inválidas:", (~va_cereales.geometry.is_valid).sum())

print("\nCRS original:")
print(va_cereales.crs)

In [ ]:
# ============================================================
# PASO 36 - REPARAR GEOMETRÍAS INVÁLIDAS
# ============================================================

from shapely import make_valid

n_antes = len(va_cereales)
invalidas_antes = (~va_cereales.geometry.is_valid).sum()

print("Recintos antes:", n_antes)
print("Geometrías inválidas antes:", invalidas_antes)

# Reparar exclusivamente las geometrías inválidas
mask_invalidas = ~va_cereales.geometry.is_valid

va_cereales.loc[
    mask_invalidas,
    "geometry"
] = va_cereales.loc[
    mask_invalidas,
    "geometry"
].apply(make_valid)

print("\nDespués de la reparación:")
print("Recintos:", len(va_cereales))
print("Geometrías nulas:", va_cereales.geometry.isna().sum())
print("Geometrías vacías:", va_cereales.geometry.is_empty.sum())
print("Geometrías inválidas:", (~va_cereales.geometry.is_valid).sum())

In [ ]:
# ============================================================
# PASO 37 - SEPARAR LOS DOS CULTIVOS
# ============================================================

va_barley = va_cereales[
    va_cereales["EC_hcat_n"] == "barley"
].copy()

va_wheat = va_cereales[
    va_cereales["EC_hcat_n"] == "common_soft_wheat"
].copy()

print("Cebada:")
print("  Recintos:", len(va_barley))
print("  Superficie (ha):", va_barley["par_supcul"].sum() / 10000)

print("\nTrigo blando:")
print("  Recintos:", len(va_wheat))
print("  Superficie (ha):", va_wheat["par_supcul"].sum() / 10000)

In [ ]:
# ============================================================
# PASO 38 - PREPARAR MÁSCARAS ESPACIALES
# ============================================================

va_barley_4326 = va_barley.to_crs("EPSG:4326")
va_wheat_4326 = va_wheat.to_crs("EPSG:4326")

print("CRS cebada:", va_barley_4326.crs)
print("CRS trigo:", va_wheat_4326.crs)

print("\nPreparadas para construir las máscaras.")

In [ ]:
# ============================================================
# PASO 39 - MUESTRA REPRODUCIBLE DE PARCELAS
# PARA VALIDACIÓN SENTINEL-2 2023
# ============================================================

N_PARCELAS_MUESTRA = 500

va_barley_sample = (
    va_barley_4326
    .sample(
        n=min(N_PARCELAS_MUESTRA, len(va_barley_4326)),
        random_state=42
    )
    .copy()
)

va_wheat_sample = (
    va_wheat_4326
    .sample(
        n=min(N_PARCELAS_MUESTRA, len(va_wheat_4326)),
        random_state=42
    )
    .copy()
)

print("Cebada - parcelas muestra:", len(va_barley_sample))
print("Trigo - parcelas muestra:", len(va_wheat_sample))

print("\nSuperficie muestra (ha):")
print(
    "Cebada:",
    va_barley_sample["par_supcul"].sum() / 10000
)
print(
    "Trigo:",
    va_wheat_sample["par_supcul"].sum() / 10000
)

In [ ]:
# ============================================================
# PASO 40 - GEODATAFRAME -> EE FEATURECOLLECTION
# ============================================================

def gdf_to_ee_featurecollection(gdf, cultivo):

    features = []

    for geom in gdf.geometry:

        feature = ee.Feature(
            ee.Geometry(geom.__geo_interface__),
            {"cultivo": cultivo}
        )

        features.append(feature)

    return ee.FeatureCollection(features)


ee_barley_sample = gdf_to_ee_featurecollection(
    va_barley_sample,
    "barley"
)

ee_wheat_sample = gdf_to_ee_featurecollection(
    va_wheat_sample,
    "common_soft_wheat"
)

print(
    "Features cebada:",
    ee_barley_sample.size().getInfo()
)

print(
    "Features trigo:",
    ee_wheat_sample.size().getInfo()
)

In [ ]:
# ============================================================
# PASO 41 - MÁSCARA DE NUBES SENTINEL-2
# ============================================================

def mask_s2_clouds(image):

    scl = image.select("SCL")

    mask = (
        scl.neq(1)   # saturated / defective
        .And(scl.neq(3))   # cloud shadow
        .And(scl.neq(8))   # medium probability cloud
        .And(scl.neq(9))   # high probability cloud
        .And(scl.neq(10))  # cirrus
        .And(scl.neq(11))  # snow / ice
    )

    return (
        image
        .updateMask(mask)
        .copyProperties(
            image,
            ["system:time_start"]
        )
    )

In [ ]:
# ============================================================
# PASO 42 - AÑADIR ÍNDICES ESPECTRALES
# ============================================================

def add_indices(image):

    # Sentinel-2 SR está escalado por 10000
    nir = image.select("B8").multiply(0.0001)
    red = image.select("B4").multiply(0.0001)
    blue = image.select("B2").multiply(0.0001)
    swir = image.select("B11").multiply(0.0001)

    ndvi = nir.subtract(red).divide(
        nir.add(red)
    ).rename("NDVI")

    evi = (
        nir.subtract(red)
        .multiply(2.5)
        .divide(
            nir
            .add(red.multiply(6))
            .subtract(blue.multiply(7.5))
            .add(1)
        )
        .rename("EVI")
    )

    ndmi = nir.subtract(swir).divide(
        nir.add(swir)
    ).rename("NDMI")

    return image.addBands(
        [ndvi, evi, ndmi]
    )

In [ ]:
# ============================================================
# PASO 43 - COLECCIÓN SENTINEL-2 PILOTO
# ============================================================

s2_piloto_limpio = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(ee_provincia_piloto)
    .filterDate(
        "2022-09-01",
        "2023-09-01"
    )
    .filter(
        ee.Filter.lte(
            "CLOUDY_PIXEL_PERCENTAGE",
            20
        )
    )
    .map(mask_s2_clouds)
    .map(add_indices)
)

print(
    "Imágenes después del prefiltro:",
    s2_piloto_limpio.size().getInfo()
)

In [ ]:
# ============================================================
# PASO 44 - COMPOSITE MARZO-ABRIL 2023
# ============================================================

s2_mar_abr = (
    s2_piloto_limpio
    .filterDate(
        "2023-03-01",
        "2023-05-01"
    )
)

print(
    "Imágenes marzo-abril:",
    s2_mar_abr.size().getInfo()
)

composite_mar_abr = (
    s2_mar_abr
    .select(
        ["NDVI", "EVI", "NDMI"]
    )
    .median()
)

print("Composite preparado.")

In [ ]:
# ============================================================
# PASO 45 - EXTRAER ÍNDICES POR PARCELA
# ============================================================

stats_barley = composite_mar_abr.reduceRegions(
    collection=ee_barley_sample,
    reducer=ee.Reducer.mean(),
    scale=20,
    tileScale=4
)

stats_wheat = composite_mar_abr.reduceRegions(
    collection=ee_wheat_sample,
    reducer=ee.Reducer.mean(),
    scale=20,
    tileScale=4
)

print("Reducciones preparadas.")

In [ ]:
# ============================================================
# PASO 46 - INSPECCIONAR RESULTADOS ESPECTRALES
# SOLO 10 PARCELAS POR CULTIVO
# ============================================================

muestra_barley = (
    stats_barley
    .limit(10)
    .getInfo()
)

muestra_wheat = (
    stats_wheat
    .limit(10)
    .getInfo()
)

print("CEBADA:")
for f in muestra_barley["features"]:
    print(f["properties"])

print("\nTRIGO BLANDO:")
for f in muestra_wheat["features"]:
    print(f["properties"])

In [ ]:
# ============================================================
# PASO 47 - TABLA DE CONTROL
# ============================================================

import pandas as pd

def features_to_df(info):
    return pd.DataFrame(
        [f["properties"] for f in info["features"]]
    )

df_barley_test = features_to_df(muestra_barley)
df_wheat_test = features_to_df(muestra_wheat)

print("CEBADA")
display(df_barley_test)

print("\nTRIGO BLANDO")
display(df_wheat_test)

In [ ]:
# ============================================================
# PASO 48 - EXTRAER RESULTADOS DE LAS 500 PARCELAS
# ============================================================

def ee_stats_to_dataframe(feature_collection):

    info = feature_collection.getInfo()

    registros = []

    for feature in info["features"]:

        p = feature["properties"]

        registros.append({
            "cultivo": p.get("cultivo"),
            "NDVI": p.get("NDVI"),
            "EVI": p.get("EVI"),
            "NDMI": p.get("NDMI")
        })

    return pd.DataFrame(registros)


df_barley = ee_stats_to_dataframe(stats_barley)
df_wheat = ee_stats_to_dataframe(stats_wheat)

print("Cebada:", df_barley.shape)
print("Trigo:", df_wheat.shape)

print("\nValores nulos - cebada:")
print(df_barley.isna().sum())

print("\nValores nulos - trigo:")
print(df_wheat.isna().sum())

In [ ]:
# ============================================================
# PASO 49 - CONTROL ESTADÍSTICO DE ÍNDICES
# ============================================================

df_indices_piloto = pd.concat(
    [df_barley, df_wheat],
    ignore_index=True
)

print("Dimensiones:", df_indices_piloto.shape)

print("\nResumen por cultivo:")

resumen_indices = (
    df_indices_piloto
    .groupby("cultivo")[["NDVI", "EVI", "NDMI"]]
    .agg([
        "count",
        "mean",
        "median",
        "std",
        "min",
        "max"
    ])
)

display(resumen_indices)

In [ ]:
# ============================================================
# PASO 50 - CONTROL DE RANGOS ESPECTRALES
# ============================================================

for indice in ["NDVI", "EVI", "NDMI"]:

    print("\n", "=" * 50)
    print(indice)
    print("=" * 50)

    print(
        df_indices_piloto
        .groupby("cultivo")[indice]
        .describe()
    )

    fuera_rango = (
        (df_indices_piloto[indice] < -1) |
        (df_indices_piloto[indice] > 1)
    ).sum()

    print(
        "\nValores fuera de [-1, 1]:",
        fuera_rango
    )

In [ ]:
# ============================================================
# PASO 51 - GUARDAR VALIDACIÓN PILOTO
# ============================================================

ruta_piloto = os.path.join(
    RUTA_PROCESADOS,
    "Sentinel2_piloto_Valladolid_2023_marzo_abril.csv"
)

df_indices_piloto.to_csv(
    ruta_piloto,
    index=False
)

print("Archivo guardado:")
print(ruta_piloto)

print("\nExiste:", os.path.exists(ruta_piloto))
print("Dimensiones:", df_indices_piloto.shape)

In [ ]:
# ============================================================
# PASO 52 - CONTROL FINAL DEL PILOTO SENTINEL-2
# ============================================================

import os
import pandas as pd

ruta_piloto = os.path.join(
    RUTA_PROCESADOS,
    "Sentinel2_piloto_Valladolid_2023_marzo_abril.csv"
)

print("Existe:", os.path.exists(ruta_piloto))

if os.path.exists(ruta_piloto):

    df_check = pd.read_csv(ruta_piloto)

    print("Dimensiones:", df_check.shape)
    print("\nRegistros por cultivo:")
    print(df_check["cultivo"].value_counts())

    print("\nValores nulos:")
    print(df_check.isna().sum())

    print("\nPrimeras filas:")
    display(df_check.head())

In [ ]:
# ============================================================
# PASO 53 CORREGIDO - VENTANAS DE CAMPAÑA
# ============================================================

def ventanas_campana(anio_cosecha):

    return {
        "nov_dec": (
            f"{anio_cosecha - 1}-11-01",
            f"{anio_cosecha}-01-01"
        ),
        "jan_feb": (
            f"{anio_cosecha}-01-01",
            f"{anio_cosecha}-03-01"
        ),
        "mar_apr": (
            f"{anio_cosecha}-03-01",
            f"{anio_cosecha}-05-01"
        ),
        "may_jun": (
            f"{anio_cosecha}-05-01",
            f"{anio_cosecha}-07-01"
        ),
        "jul": (
            f"{anio_cosecha}-07-01",
            f"{anio_cosecha}-08-01"
        )
    }


ventanas_2023 = ventanas_campana(2023)

print("VENTANAS CAMPAÑA 2023")
print("=" * 45)

for nombre, (inicio, fin) in ventanas_2023.items():
    print(f"{nombre:10s}: {inicio} -> {fin}")

In [ ]:
# ============================================================
# PASO 54 CORREGIDO - DISPONIBILIDAD POR VENTANA
# VALLADOLID - CAMPAÑA 2023
# ============================================================

resultados_disponibilidad = []

for ventana, (fecha_inicio, fecha_fin) in ventanas_2023.items():

    coleccion = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(ee_provincia_piloto)
        .filterDate(fecha_inicio, fecha_fin)
    )

    n_total = coleccion.size().getInfo()

    n_cloud20 = (
        coleccion
        .filter(
            ee.Filter.lte(
                "CLOUDY_PIXEL_PERCENTAGE",
                20
            )
        )
        .size()
        .getInfo()
    )

    resultados_disponibilidad.append({
        "ventana": ventana,
        "fecha_inicio": fecha_inicio,
        "fecha_fin": fecha_fin,
        "n_total": n_total,
        "n_cloud20": n_cloud20
    })


df_disponibilidad = pd.DataFrame(
    resultados_disponibilidad
)

display(df_disponibilidad)

print(
    "\nTotal imágenes:",
    df_disponibilidad["n_total"].sum()
)

print(
    "\nTotal con nubosidad <=20 %:",
    df_disponibilidad["n_cloud20"].sum()
)

In [ ]:
# ============================================================
# PASO 55 - FUNCIONES SENTINEL-2 Y COMPOSITES TEMPORALES
# ============================================================

def mask_s2_scl(image):
    """
    Máscara de píxeles no válidos mediante la banda SCL.

    Se eliminan:
    0  = No data
    1  = Saturated / defective
    3  = Cloud shadow
    8  = Cloud medium probability
    9  = Cloud high probability
    10 = Thin cirrus
    11 = Snow / ice
    """

    scl = image.select("SCL")

    mask = (
        scl.neq(0)
        .And(scl.neq(1))
        .And(scl.neq(3))
        .And(scl.neq(8))
        .And(scl.neq(9))
        .And(scl.neq(10))
        .And(scl.neq(11))
    )

    return image.updateMask(mask)


def add_indices(image):

    # Sentinel-2 SR está escalado por 10000
    nir = image.select("B8").multiply(0.0001)
    red = image.select("B4").multiply(0.0001)
    blue = image.select("B2").multiply(0.0001)
    swir = image.select("B11").multiply(0.0001)

    ndvi = (
        nir.subtract(red)
        .divide(nir.add(red))
        .rename("NDVI")
    )

    ndmi = (
        nir.subtract(swir)
        .divide(nir.add(swir))
        .rename("NDMI")
    )

    evi = (
        nir.subtract(red)
        .multiply(2.5)
        .divide(
            nir
            .add(red.multiply(6))
            .subtract(blue.multiply(7.5))
            .add(1)
        )
        .rename("EVI")
    )

    return image.addBands([ndvi, evi, ndmi])


def crear_composite(fecha_inicio, fecha_fin, geometria):

    coleccion = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(geometria)
        .filterDate(fecha_inicio, fecha_fin)
        .filter(
            ee.Filter.lte(
                "CLOUDY_PIXEL_PERCENTAGE",
                20
            )
        )
        .map(mask_s2_scl)
        .map(add_indices)
    )

    composite = (
        coleccion
        .select(["NDVI", "EVI", "NDMI"])
        .median()
    )

    return composite, coleccion.size()

In [ ]:
# ============================================================
# CREAR COMPOSITES - VALLADOLID 2023
# ============================================================

composites_2023 = {}

for ventana, (inicio, fin) in ventanas_2023.items():

    composite, n_img = crear_composite(
        inicio,
        fin,
        ee_provincia_piloto
    )

    composites_2023[ventana] = composite

    print(
        ventana,
        "->",
        n_img.getInfo(),
        "imágenes"
    )

print("\nComposites preparados:", len(composites_2023))

In [ ]:
# ============================================================
# PASO 56 CORREGIDO - CONTROL RÁPIDO DE COMPOSITES
# ============================================================

resultados_control = []

for ventana, composite in composites_2023.items():

    print("Procesando:", ventana)

    stats = composite.reduceRegion(
        reducer=ee.Reducer.mean()
            .combine(
                reducer2=ee.Reducer.median(),
                sharedInputs=True
            ),
        geometry=ee_provincia_piloto,
        scale=500,          # solo diagnóstico
        bestEffort=True,
        maxPixels=1e7,
        tileScale=4
    ).getInfo()

    resultados_control.append({
        "ventana": ventana,
        "NDVI_mean": stats.get("NDVI_mean"),
        "NDVI_median": stats.get("NDVI_median"),
        "EVI_mean": stats.get("EVI_mean"),
        "EVI_median": stats.get("EVI_median"),
        "NDMI_mean": stats.get("NDMI_mean"),
        "NDMI_median": stats.get("NDMI_median")
    })


df_control_composites = pd.DataFrame(resultados_control)

display(df_control_composites)

In [ ]:
# ============================================================
# PASO 57 - COMPROBAR VARIABLES DISPONIBLES DEL PILOTO
# ============================================================

variables_comprobar = [
    "gdf_valladolid",
    "valladolid",
    "gdf_cultivos",
    "gdf_piloto"
]

for var in variables_comprobar:
    if var in globals():
        obj = globals()[var]

        try:
            print(
                var,
                "->",
                type(obj),
                "dimensiones:",
                obj.shape
            )
        except:
            print(var, "->", type(obj))

In [ ]:
# ============================================================
# PASO 58 - RECARGAR EUROCROPS DE VALLADOLID
# ============================================================

import geopandas as gpd
import os

ruta_va = os.path.join(
    RUTA_EUROCROPS,
    "ES_2023",
    "es_va_2023",
    "es_va_2023.shp"
)

va_cereales = gpd.read_file(
    ruta_va,
    columns=[
        "EC_hcat_n",
        "par_supcul",
        "geometry"
    ]
)

va_cereales = va_cereales[
    va_cereales["EC_hcat_n"].isin(
        ["barley", "common_soft_wheat"]
    )
].copy()

print("Dimensiones:", va_cereales.shape)

print("\nRecintos por cultivo:")
print(
    va_cereales["EC_hcat_n"].value_counts()
)

print("\nCRS:")
print(va_cereales.crs)

In [ ]:
# ============================================================
# PASO 59 - REPARAR Y PASAR A EPSG:4326
# ============================================================

from shapely import make_valid

mask_invalidas = ~va_cereales.geometry.is_valid

va_cereales.loc[
    mask_invalidas,
    "geometry"
] = va_cereales.loc[
    mask_invalidas,
    "geometry"
].apply(make_valid)

print(
    "Geometrías inválidas:",
    (~va_cereales.geometry.is_valid).sum()
)

va_cereales = va_cereales.to_crs("EPSG:4326")

print("CRS final:", va_cereales.crs)

In [ ]:
# ============================================================
# PASO 60 - MUESTRAS PILOTO
# ============================================================

N = 500

va_barley_sample = (
    va_cereales[
        va_cereales["EC_hcat_n"] == "barley"
    ]
    .sample(n=N, random_state=42)
    .copy()
)

va_wheat_sample = (
    va_cereales[
        va_cereales["EC_hcat_n"] == "common_soft_wheat"
    ]
    .sample(n=N, random_state=42)
    .copy()
)

print("Cebada:", len(va_barley_sample))
print("Trigo:", len(va_wheat_sample))

In [ ]:
# ============================================================
# PASO 61 - CONVERTIR MUESTRAS A GEOMETRÍAS EARTH ENGINE
# ============================================================

from shapely.geometry import mapping

def geodataframe_a_ee_geometry(gdf):
    """
    Une las geometrías del GeoDataFrame y las convierte
    en una geometría compatible con Earth Engine.
    """

    geom_union = gdf.geometry.union_all()

    return ee.Geometry(mapping(geom_union))


ee_barley_sample = geodataframe_a_ee_geometry(
    va_barley_sample
)

ee_wheat_sample = geodataframe_a_ee_geometry(
    va_wheat_sample
)

print("Geometría cebada preparada.")
print("Geometría trigo blando preparada.")
print("Número de recintos originales:")
print("  Cebada:", len(va_barley_sample))
print("  Trigo :", len(va_wheat_sample))

In [ ]:
# ============================================================
# PASO 62 - TRAYECTORIA SENTINEL-2 POR CULTIVO
# VALLADOLID - CAMPAÑA 2023
# ============================================================

resultados_trayectoria = []

geometrias_cultivo = {
    "barley": ee_barley_sample,
    "common_soft_wheat": ee_wheat_sample
}

for cultivo, geometria in geometrias_cultivo.items():

    print("\nProcesando:", cultivo)

    for ventana, composite in composites_2023.items():

        print("   ", ventana)

        stats = composite.reduceRegion(
            reducer=ee.Reducer.median(),
            geometry=geometria,
            scale=20,
            maxPixels=1e8,
            tileScale=4
        ).getInfo()

        resultados_trayectoria.append({
            "cultivo": cultivo,
            "ventana": ventana,
            "NDVI": stats.get("NDVI"),
            "EVI": stats.get("EVI"),
            "NDMI": stats.get("NDMI")
        })


df_trayectoria = pd.DataFrame(
    resultados_trayectoria
)

display(df_trayectoria)

In [ ]:
# ============================================================
# PASO 63 - CONTROL DE RESULTADOS
# ============================================================

print("Dimensiones:", df_trayectoria.shape)

print("\nValores nulos:")
print(df_trayectoria.isna().sum())

print("\nRangos:")

for indice in ["NDVI", "EVI", "NDMI"]:

    print(f"\n{indice}")

    print(
        "mínimo:",
        df_trayectoria[indice].min()
    )

    print(
        "máximo:",
        df_trayectoria[indice].max()
    )

    print(
        "fuera [-1,1]:",
        (
            (df_trayectoria[indice] < -1) |
            (df_trayectoria[indice] > 1)
        ).sum()
    )

In [ ]:

# ============================================================
# PASO 64 - GUARDAR VALIDACIÓN PILOTO SENTINEL-2
# ============================================================

import os

ruta_validacion = os.path.join(
    RUTA_PROCESADOS,
    "Sentinel2_validacion_Valladolid_2023_ventanas.csv"
)

df_trayectoria.to_csv(
    ruta_validacion,
    index=False
)

print("Archivo guardado:")
print(ruta_validacion)

print("\nExiste:", os.path.exists(ruta_validacion))
print("Dimensiones:", df_trayectoria.shape)

display(df_trayectoria)

# Decidir como avanzar

In [ ]:
# ============================================================
# PASO 65 - INVENTARIO LOCAL DE EUROCROPS
# ============================================================

import os
import glob

RUTA_EUROCROPS = (
    "/content/TFM_Rendimiento_cultivos/"
    "01_Datos_originales/EuroCrops"
)

print("=" * 65)
print("INVENTARIO LOCAL EUROCROPS")
print("=" * 65)

# Carpetas principales
carpetas = sorted([
    x for x in glob.glob(os.path.join(RUTA_EUROCROPS, "*"))
    if os.path.isdir(x)
])

print("\nCARPETAS PRINCIPALES:")
for carpeta in carpetas:
    print(" -", os.path.basename(carpeta))

# Buscar todos los shapefiles
shapefiles = glob.glob(
    os.path.join(RUTA_EUROCROPS, "**", "*.shp"),
    recursive=True
)

print("\nNúmero total de shapefiles:", len(shapefiles))

# Agrupar aproximadamente por año según la ruta
for ano in range(2017, 2024):

    archivos_ano = [
        f for f in shapefiles
        if str(ano) in f
    ]

    print(
        f"{ano}: "
        f"{len(archivos_ano)} shapefiles"
    )

print("\nEjemplos de archivos:")
for f in shapefiles[:10]:
    print(" -", f)

In [ ]:
# Comprobar Dynamic World

In [ ]:
# ============================================================
# PASO 66 DEFINITIVO - COMPROBAR DYNAMIC WORLD
# ============================================================

dw_test = (
    ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
    .filterBounds(ee_provincia_piloto)
    .filterDate("2023-03-01", "2023-05-01")
)

print(
    "Imágenes Dynamic World "
    "Valladolid marzo-abril 2023:",
    dw_test.size().getInfo()
)

primera_dw = ee.Image(dw_test.first())

print("\nBandas Dynamic World:")
print(primera_dw.bandNames().getInfo())

In [ ]:
# ============================================================
# PASO 67 - MÁSCARA AGRÍCOLA DINÁMICA
# ============================================================

def mascara_cultivos_dynamic_world(
    fecha_inicio,
    fecha_fin,
    geometria,
    threshold=0.5
):

    dw = (
        ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
        .filterBounds(geometria)
        .filterDate(fecha_inicio, fecha_fin)
        .select("crops")
    )

    # Probabilidad mediana de clase crops
    prob_crops = dw.median()

    # Máscara agrícola
    mask_crops = prob_crops.gte(threshold)

    return mask_crops

In [ ]:
# ============================================================
# PASO 68 - COMPOSITE SENTINEL-2 AGRÍCOLA
# ============================================================

def composite_s2_agricola(
    fecha_inicio,
    fecha_fin,
    geometria
):

    # ----------------------------------------
    # Máscara agrícola dinámica
    # ----------------------------------------

    mask_crops = mascara_cultivos_dynamic_world(
        fecha_inicio,
        fecha_fin,
        geometria,
        threshold=0.5
    )

    # ----------------------------------------
    # Sentinel-2
    # ----------------------------------------

    s2 = (
        ee.ImageCollection(
            "COPERNICUS/S2_SR_HARMONIZED"
        )
        .filterBounds(geometria)
        .filterDate(
            fecha_inicio,
            fecha_fin
        )
        .filter(
            ee.Filter.lte(
                "CLOUDY_PIXEL_PERCENTAGE",
                20
            )
        )
        .map(mask_s2_scl)
        .map(add_indices)
    )

    composite = (
        s2
        .select(
            ["NDVI", "EVI", "NDMI"]
        )
        .median()
        .updateMask(mask_crops)
    )

    return composite

In [ ]:
# ============================================================
# PASO 69 - 50 PROVINCIAS -> EARTH ENGINE
# ============================================================

features_provincias = []

for _, fila in provincias.iterrows():

    geom = fila.geometry

    # Simplificación ligera para reducir complejidad
    geom_simple = geom.simplify(
        tolerance=0.001,
        preserve_topology=True
    )

    feature = ee.Feature(
        ee.Geometry(
            geom_simple.__geo_interface__
        ),
        {
            "provincia_estandar":
                fila["provincia_estandar"]
        }
    )

    features_provincias.append(feature)


ee_provincias = ee.FeatureCollection(
    features_provincias
)

print(
    "Provincias en Earth Engine:",
    ee_provincias.size().getInfo()
)

In [ ]:
# ============================================================
# PASO 70 - PROCESAR UNA CAMPAÑA SENTINEL-2
# 50 PROVINCIAS
# ============================================================

def procesar_campana_sentinel(anio_cosecha):

    ventanas = ventanas_campana(anio_cosecha)

    resultados = []

    for ventana, (inicio, fin) in ventanas.items():

        print(
            f"Procesando {anio_cosecha} | {ventana} "
            f"| {inicio} -> {fin}"
        )

        composite = composite_s2_agricola(
            inicio,
            fin,
            ee_espana
        )

        stats = composite.reduceRegions(
            collection=ee_provincias,
            reducer=ee.Reducer.median(),
            scale=100,
            tileScale=4
        )

        stats = stats.map(
            lambda f: f.set({
                "ano": anio_cosecha,
                "ventana": ventana
            })
        )

        resultados.append(stats)

    return ee.FeatureCollection(
        resultados
    ).flatten()

# Procesamiento de datos de los 7 años para las 50 provincias - Definitivo TFM

In [ ]:
# ============================================================
# PASO 71 - CARPETA PARA RESULTADOS SENTINEL-2
# ============================================================

RUTA_SENTINEL_PROCESADO = os.path.join(
    RUTA_PROCESADOS,
    "Sentinel2"
)

os.makedirs(
    RUTA_SENTINEL_PROCESADO,
    exist_ok=True
)

print("Carpeta:")
print(RUTA_SENTINEL_PROCESADO)
print("Existe:", os.path.exists(RUTA_SENTINEL_PROCESADO))

In [ ]:
# ============================================================
# PASO 72 - SENTINEL-2 NACIONAL 2017-2023
# 50 provincias × 5 ventanas
# ============================================================

resultados_sentinel = []

for ano in range(2017, 2024):

    print("\n" + "=" * 70)
    print("CAMPAÑA", ano)
    print("=" * 70)

    try:

        fc_ano = procesar_campana_sentinel(
            ano
        )

        print("Procesamiento preparado.")

        # Traer únicamente las propiedades
        info = fc_ano.getInfo()

        registros = []

        for feature in info["features"]:

            p = feature["properties"]

            registros.append({
                "ano": p.get("ano"),
                "provincia_estandar":
                    p.get("provincia_estandar"),
                "ventana": p.get("ventana"),
                "NDVI": p.get("NDVI"),
                "EVI": p.get("EVI"),
                "NDMI": p.get("NDMI")
            })

        df_ano = pd.DataFrame(
            registros
        )

        print(
            "Registros obtenidos:",
            len(df_ano)
        )

        # Guardar inmediatamente en Drive
        ruta_ano = os.path.join(
            RUTA_SENTINEL_PROCESADO,
            f"Sentinel2_provincial_{ano}.csv"
        )

        df_ano.to_csv(
            ruta_ano,
            index=False,
            encoding="utf-8-sig"
        )

        print("Guardado:")
        print(ruta_ano)

        resultados_sentinel.append(
            df_ano
        )

    except Exception as e:

        print("\nERROR EN", ano)
        print(e)

        # No detener los demás años
        continue

In [ ]:
# ============================================================
# PASO 73 - UNIR RESULTADOS SENTINEL-2
# ============================================================

if len(resultados_sentinel) > 0:

    sentinel_long = pd.concat(
        resultados_sentinel,
        ignore_index=True
    )

    print(
        "Dimensiones:",
        sentinel_long.shape
    )

    print("\nRegistros por año:")
    print(
        sentinel_long.groupby("ano").size()
    )

    print("\nValores nulos:")
    print(
        sentinel_long[
            ["NDVI", "EVI", "NDMI"]
        ].isna().sum()
    )

    display(
        sentinel_long.head(20)
    )

In [ ]:
# ============================================================
# PASO 74 - GUARDAR SENTINEL-2 NACIONAL
# ============================================================

ruta_sentinel_long = os.path.join(
    RUTA_PROCESADOS,
    "Sentinel2_provincial_2017_2023_long.csv"
)

sentinel_long.to_csv(
    ruta_sentinel_long,
    index=False,
    encoding="utf-8-sig"
)

print("Guardado:")
print(ruta_sentinel_long)

print(
    "\nDimensiones:",
    sentinel_long.shape
)

In [ ]:
# ============================================================
# PASO 75 - AUDITORÍA TEMPORAL SENTINEL-2 SR 2017
# ============================================================

ventanas_2017 = ventanas_campana(2017)

auditoria_2017 = []

for ventana, (inicio, fin) in ventanas_2017.items():

    coleccion = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(ee_espana)
        .filterDate(inicio, fin)
    )

    n_total = coleccion.size().getInfo()

    coleccion_cloud20 = coleccion.filter(
        ee.Filter.lte(
            "CLOUDY_PIXEL_PERCENTAGE",
            20
        )
    )

    n_cloud20 = coleccion_cloud20.size().getInfo()

    auditoria_2017.append({
        "ventana": ventana,
        "inicio": inicio,
        "fin": fin,
        "n_total": n_total,
        "n_cloud20": n_cloud20
    })

auditoria_2017_df = pd.DataFrame(auditoria_2017)

display(auditoria_2017_df)

In [ ]:
# ============================================================
# PASO 76 - LOCALIZAR NULOS
# ============================================================

nulos_sentinel = sentinel_long[
    sentinel_long[
        ["NDVI", "EVI", "NDMI"]
    ].isna().any(axis=1)
].copy()

print("Registros con algún índice nulo:")
print(len(nulos_sentinel))

display(
    nulos_sentinel.sort_values(
        ["ano", "ventana", "provincia_estandar"]
    )
)

In [ ]:
# ============================================================
# PASO 77 - AUDITORÍA DEFINITIVA SENTINEL-2
# ============================================================

pruebas = [
    ("SR_nov_dec", "COPERNICUS/S2_SR_HARMONIZED",
     "2016-11-01", "2017-01-01"),

    ("SR_jan_feb", "COPERNICUS/S2_SR_HARMONIZED",
     "2017-01-01", "2017-03-01"),

    ("SR_mar_apr", "COPERNICUS/S2_SR_HARMONIZED",
     "2017-03-01", "2017-05-01"),

    ("L1C_nov_dec", "COPERNICUS/S2_HARMONIZED",
     "2016-11-01", "2017-01-01"),

    ("L1C_jan_feb", "COPERNICUS/S2_HARMONIZED",
     "2017-01-01", "2017-03-01")
]

resultados = []

for nombre, coleccion_id, inicio, fin in pruebas:

    col = (
        ee.ImageCollection(coleccion_id)
        .filterDate(inicio, fin)
        .filterBounds(ee_espana)
    )

    resultados.append({
        "prueba": nombre,
        "coleccion": coleccion_id,
        "inicio": inicio,
        "fin": fin,
        "n_imagenes": col.size().getInfo()
    })

df_auditoria = pd.DataFrame(resultados)

display(df_auditoria)

In [ ]:
# ============================================================
# PASO 78 - COMPROBAR FECHAS REALES S2 SR
# ============================================================

def comprobar_fechas_s2(inicio, fin, nombre):

    col = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(ee_espana)
        .filterDate(inicio, fin)
        .sort("system:time_start")
    )

    n = col.size().getInfo()

    print("\n" + "=" * 60)
    print(nombre)
    print("=" * 60)
    print("Intervalo solicitado:", inicio, "->", fin)
    print("Número de imágenes:", n)

    if n > 0:
        primera = ee.Image(col.first())
        ultima = ee.Image(col.sort("system:time_start", False).first())

        fecha_primera = ee.Date(
            primera.get("system:time_start")
        ).format("YYYY-MM-dd HH:mm:ss").getInfo()

        fecha_ultima = ee.Date(
            ultima.get("system:time_start")
        ).format("YYYY-MM-dd HH:mm:ss").getInfo()

        print("Primera imagen:", fecha_primera)
        print("Última imagen:", fecha_ultima)

        # Mostrar las primeras 10 fechas
        fechas = (
            col.limit(10)
            .aggregate_array("system:time_start")
            .getInfo()
        )

        fechas_legibles = [
            ee.Date(f).format("YYYY-MM-dd HH:mm:ss").getInfo()
            for f in fechas
        ]

        print("\nPrimeras fechas:")
        for f in fechas_legibles:
            print("  ", f)


comprobar_fechas_s2(
    "2016-11-01",
    "2017-01-01",
    "nov_dec campaña 2017"
)

comprobar_fechas_s2(
    "2017-01-01",
    "2017-03-01",
    "jan_feb campaña 2017"
)

comprobar_fechas_s2(
    "2017-03-01",
    "2017-05-01",
    "mar_apr campaña 2017"
)

In [ ]:
# ============================================================
# PASO 79 - GUARDAR SENTINEL-2 NACIONAL EN FORMATO LONG
# ============================================================

ruta_sentinel_long = os.path.join(
    RUTA_PROCESADOS,
    "Sentinel2_provincial_2017_2023_long.csv"
)

sentinel_long.to_csv(
    ruta_sentinel_long,
    index=False,
    encoding="utf-8-sig"
)

print("Archivo guardado:")
print(ruta_sentinel_long)

print("\nExiste:", os.path.exists(ruta_sentinel_long))
print("Dimensiones:", sentinel_long.shape)

In [ ]:
# ============================================================
# PASO 80 - SENTINEL-2 EN FORMATO ANCHO
# 1 fila = provincia × año
# ============================================================

sentinel_wide = sentinel_long.pivot(
    index=[
        "ano",
        "provincia_estandar"
    ],
    columns="ventana",
    values=[
        "NDVI",
        "EVI",
        "NDMI"
    ]
)

# Convertir columnas multinivel a nombres simples
sentinel_wide.columns = [
    f"{indice}_{ventana}"
    for indice, ventana
    in sentinel_wide.columns
]

sentinel_wide = (
    sentinel_wide
    .reset_index()
    .sort_values(
        ["ano", "provincia_estandar"]
    )
    .reset_index(drop=True)
)

print("Dimensiones:")
print(sentinel_wide.shape)

print("\nProvincias:")
print(
    sentinel_wide[
        "provincia_estandar"
    ].nunique()
)

print("\nAños:")
print(
    sorted(
        sentinel_wide["ano"].unique()
    )
)

print("\nColumnas:")
print(sentinel_wide.columns.tolist())

display(sentinel_wide.head())

In [ ]:
# ============================================================
# PASO 81 - AUDITORÍA FINAL DE VALORES NULOS
# ============================================================

columnas_sentinel = [
    c for c in sentinel_wide.columns
    if c.startswith(
        ("NDVI_", "EVI_", "NDMI_")
    )
]

print("Nulos por variable:")
print(
    sentinel_wide[
        columnas_sentinel
    ]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

filas_con_nulos = sentinel_wide[
    sentinel_wide[
        columnas_sentinel
    ]
    .isna()
    .any(axis=1)
].copy()

print(
    "\nNúmero de provincia-año "
    "con algún dato Sentinel-2 ausente:",
    len(filas_con_nulos)
)

display(
    filas_con_nulos[
        [
            "ano",
            "provincia_estandar"
        ] + columnas_sentinel
    ]
)

In [ ]:
# ============================================================
# PASO 82 - GUARDAR DATASET SENTINEL-2 DEFINITIVO
# ============================================================

ruta_sentinel_final = os.path.join(
    RUTA_PROCESADOS,
    "Sentinel2_provincial_2017_2023_features.csv"
)

sentinel_wide.to_csv(
    ruta_sentinel_final,
    index=False,
    encoding="utf-8-sig"
)

print("Archivo guardado:")
print(ruta_sentinel_final)

print("\nExiste:", os.path.exists(ruta_sentinel_final))
print("Dimensiones:", sentinel_wide.shape)

# Dataset final Sentinel

In [ ]:
# ============================================================
# PASO 83 - CARGAR BLOQUES PARA DATASET MAESTRO
# ============================================================

import os
import pandas as pd

# ------------------------------------------------------------
# Rutas
# ------------------------------------------------------------

RUTA_PROCESADOS = (
    "/content/TFM_Rendimiento_cultivos/"
    "TFM_Rendimiento_cultivos/"
    "02_Datos_procesados"
)

# TARGET MAPA
ruta_target = os.path.join(
    RUTA_PROCESADOS,
    "target_trigo_cebada_2017_2023.csv"
)

# ERA5-Land
ruta_era5 = os.path.join(
    RUTA_PROCESADOS,
    "ERA5Land_features_provincia_campana_2017_2023.csv"
)

# SoilGrids
ruta_soil = os.path.join(
    RUTA_PROCESADOS,
    "SoilGrids_provincial_0_30cm.csv"
)

# EuroCrops
ruta_eurocrops = os.path.join(
    RUTA_PROCESADOS,
    "EuroCrops_2023_trigo_cebada_provincial.csv"
)

# Sentinel-2
ruta_sentinel = os.path.join(
    RUTA_PROCESADOS,
    "Sentinel2_provincial_2017_2023_features.csv"
)

# ------------------------------------------------------------
# Cargar
# ------------------------------------------------------------

target = pd.read_csv(ruta_target)
era5 = pd.read_csv(ruta_era5)
soil = pd.read_csv(ruta_soil)
eurocrops = pd.read_csv(ruta_eurocrops)
sentinel = pd.read_csv(ruta_sentinel)

# ------------------------------------------------------------
# Control
# ------------------------------------------------------------

bloques = {
    "TARGET_MAPA": target,
    "ERA5": era5,
    "SOILGRIDS": soil,
    "EUROCROPS": eurocrops,
    "SENTINEL2": sentinel
}

for nombre, df in bloques.items():

    print("\n" + "=" * 70)
    print(nombre)
    print("=" * 70)

    print("Dimensiones:", df.shape)
    print("Columnas:")
    print(df.columns.tolist())

    print("\nPrimeras filas:")
    display(df.head(3))

In [ ]:
# ============================================================
# PASO 84 - PREPARAR CLAVES
# ============================================================

era5_merge = era5.rename(
    columns={"ano_campana": "ano"}
).copy()

print("ERA5 columnas clave:")
print(
    era5_merge[
        ["provincia_estandar", "ano"]
    ].head()
)

In [ ]:
# ============================================================
# PASO 85 - MERGE MAESTRO
# ============================================================

dataset_maestro = target.copy()

print("Inicio TARGET:", dataset_maestro.shape)

# ------------------------------------------------------------
# 1. ERA5 -> provincia + año
# ------------------------------------------------------------

dataset_maestro = dataset_maestro.merge(
    era5_merge,
    on=[
        "provincia_estandar",
        "ano"
    ],
    how="left",
    validate="many_to_one"
)

print("Después ERA5:", dataset_maestro.shape)

# ------------------------------------------------------------
# 2. SoilGrids -> provincia
# ------------------------------------------------------------

dataset_maestro = dataset_maestro.merge(
    soil,
    on="provincia_estandar",
    how="left",
    validate="many_to_one"
)

print("Después SoilGrids:", dataset_maestro.shape)

# ------------------------------------------------------------
# 3. Sentinel-2 -> provincia + año
# ------------------------------------------------------------

dataset_maestro = dataset_maestro.merge(
    sentinel,
    on=[
        "provincia_estandar",
        "ano"
    ],
    how="left",
    validate="many_to_one"
)

print("Después Sentinel-2:", dataset_maestro.shape)

# ------------------------------------------------------------
# 4. EuroCrops -> provincia + cultivo
# ------------------------------------------------------------

dataset_maestro = dataset_maestro.merge(
    eurocrops,
    on=[
        "provincia_estandar",
        "cultivo"
    ],
    how="left",
    validate="many_to_one"
)

print("Después EuroCrops:", dataset_maestro.shape)

In [ ]:
# ============================================================
# PASO 86 - CONTROL ESTRUCTURAL DEL DATASET MAESTRO
# ============================================================

print("Dimensiones finales:")
print(dataset_maestro.shape)

print("\nFilas esperadas:", len(target))
print("Filas obtenidas:", len(dataset_maestro))

duplicados = dataset_maestro.duplicated(
    subset=[
        "ano",
        "cultivo",
        "provincia_estandar"
    ]
).sum()

print("\nDuplicados provincia × año × cultivo:")
print(duplicados)

print("\nAños:")
print(sorted(dataset_maestro["ano"].unique()))

print("\nCultivos:")
print(dataset_maestro["cultivo"].value_counts())

In [ ]:
# ============================================================
# PASO 87 - AUDITORÍA DE NULOS
# ============================================================

nulos = (
    dataset_maestro
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("Columnas con valores nulos:")

display(
    nulos[nulos > 0].to_frame(
        "n_nulos"
    )
)

In [ ]:
# Número de observaciones afectadas por algún nulo
filas_con_nulos = dataset_maestro[
    dataset_maestro.isna().any(axis=1)
]

print(
    "Observaciones con algún valor nulo:",
    len(filas_con_nulos)
)

print(
    "Porcentaje:",
    round(
        100 * len(filas_con_nulos) /
        len(dataset_maestro),
        2
    ),
    "%"
)

In [ ]:
# ============================================================
# PASO 88 - GUARDAR DATASET MAESTRO
# ============================================================

ruta_dataset_maestro = os.path.join(
    RUTA_PROCESADOS,
    "dataset_maestro_TFM_2017_2023.csv"
)

dataset_maestro.to_csv(
    ruta_dataset_maestro,
    index=False,
    encoding="utf-8-sig"
)

print("Guardado:")
print(ruta_dataset_maestro)

print("\nExiste:", os.path.exists(ruta_dataset_maestro))
print("Dimensiones:", dataset_maestro.shape)

In [ ]:
# ============================================================
# PASO 89 - IDENTIFICAR OBSERVACIONES CON DATOS S2 AUSENTES
# ============================================================

cols_s2_nulos = [
    "NDVI_nov_dec", "EVI_nov_dec", "NDMI_nov_dec",
    "NDVI_jan_feb", "EVI_jan_feb", "NDMI_jan_feb"
]

mask_nulos_s2 = dataset_maestro[cols_s2_nulos].isna().any(axis=1)

casos_nulos_s2 = dataset_maestro.loc[
    mask_nulos_s2,
    [
        "ano",
        "provincia_estandar",
        "cultivo",
        "rendimiento_kg_ha"
    ] + cols_s2_nulos
].sort_values(
    ["ano", "provincia_estandar", "cultivo"]
)

print("Observaciones afectadas:", len(casos_nulos_s2))
print(
    "Porcentaje del dataset:",
    round(100 * len(casos_nulos_s2) / len(dataset_maestro), 2),
    "%"
)

display(casos_nulos_s2)

In [ ]:
# ============================================================
# PASO 90 - DISTRIBUCIÓN DE LOS CASOS CON NULOS
# ============================================================

print("Por año:")
print(
    casos_nulos_s2
    .groupby("ano")
    .size()
)

print("\nPor cultivo:")
print(
    casos_nulos_s2
    .groupby("cultivo")
    .size()
)

print("\nPor año y cultivo:")
display(
    casos_nulos_s2
    .groupby(["ano", "cultivo"])
    .size()
    .unstack(fill_value=0)
)

In [ ]:
# ============================================================
# PASO 91 - AUDITORÍA DE VARIABLES DEL DATASET MAESTRO
# ============================================================

import pandas as pd
import numpy as np

ruta_dataset = (
    "/content/TFM_Rendimiento_cultivos/"
    "02_Datos_procesados/dataset_maestro_TFM_2017_2023.csv"
)

df = pd.read_csv(ruta_dataset)

print("Dimensiones:", df.shape)

# ------------------------------------------------------------
# 1. LISTADO COMPLETO DE COLUMNAS
# ------------------------------------------------------------

print("\nCOLUMNAS DEL DATASET:")
for i, col in enumerate(df.columns, start=1):
    print(f"{i:02d}. {col}")


# ------------------------------------------------------------
# 2. CLASIFICACIÓN DE VARIABLES POR BLOQUE
# ------------------------------------------------------------

identificadores = [
    c for c in df.columns
    if c in ["ano", "cultivo", "provincia_estandar"]
]

target = [
    c for c in df.columns
    if c == "rendimiento_kg_ha"
]

sentinel = [
    c for c in df.columns
    if c.startswith(("NDVI_", "EVI_", "NDMI_"))
]

soilgrids = [
    c for c in df.columns
    if any(x in c for x in [
        "clay_pct",
        "sand_pct",
        "soc_g_kg",
        "ph_0_30cm",
        "bdod_g_cm3"
    ])
]

eurocrops = [
    c for c in df.columns
    if c in [
        "n_recintos",
        "superficie_eurocrops_ha",
        "superficie_media_recinto_ha"
    ]
]

# ERA5 = columnas restantes
ya_clasificadas = set(
    identificadores +
    target +
    sentinel +
    soilgrids +
    eurocrops
)

era5 = [
    c for c in df.columns
    if c not in ya_clasificadas
]


# ------------------------------------------------------------
# 3. RESUMEN
# ------------------------------------------------------------

bloques = {
    "Identificadores": identificadores,
    "Target": target,
    "ERA5": era5,
    "SoilGrids": soilgrids,
    "EuroCrops": eurocrops,
    "Sentinel-2": sentinel
}

print("\n" + "=" * 70)
print("CLASIFICACIÓN DE VARIABLES")
print("=" * 70)

total = 0

for nombre, columnas in bloques.items():

    print(f"\n{nombre}: {len(columnas)} variables")

    for c in columnas:
        print("  -", c)

    total += len(columnas)

print("\nTotal variables clasificadas:", total)
print("Total columnas dataset:", len(df.columns))


# ------------------------------------------------------------
# 4. TIPOS Y NULOS
# ------------------------------------------------------------

auditoria = pd.DataFrame({
    "variable": df.columns,
    "tipo": df.dtypes.astype(str).values,
    "n_nulos": df.isna().sum().values,
    "pct_nulos": (df.isna().mean() * 100).round(2).values,
    "n_unicos": df.nunique(dropna=True).values
})

print("\n" + "=" * 70)
print("AUDITORÍA GENERAL")
print("=" * 70)

display(auditoria)


# ------------------------------------------------------------
# 5. VARIABLES CONSTANTES O CASI CONSTANTES
# ------------------------------------------------------------

numericas = df.select_dtypes(include=np.number)

print("\nVariables numéricas:", numericas.shape[1])

constantes = [
    c for c in numericas.columns
    if numericas[c].nunique(dropna=True) <= 1
]

print("\nVariables constantes:")
print(constantes if constantes else "Ninguna")


# ------------------------------------------------------------
# 6. CONTROL FINAL DE MUESTRAS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MUESTRAS POR CULTIVO")
print("=" * 70)

print(df.groupby("cultivo").size())

print("\nMuestras por año y cultivo:")
display(
    pd.crosstab(
        df["ano"],
        df["cultivo"]
    )
)